In [8]:
# E1: SparkSession Personalizada:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .appName("MiPrimerAnalisis")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
spark   # muestra la info de la sesión
print("App:", spark.sparkContext.appName)
print("Executor memory:", spark.conf.get("spark.executor.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

App: MiPrimerAnalisis
Executor memory: 1g
Shuffle partitions: 8


In [9]:
spark

In [10]:
# E2: Operaciones Basicas:

sc = spark.sparkContext
numeros = list(range(1, 21))
rdd = sc.parallelize(numeros)
print("Total de elementos:", rdd.count())     # 20
print("Primeros 5:", rdd.take(5))              # [1, 2, 3, 4, 5]
print("Suma:", rdd.sum())                      # 210
print("Máximo:", rdd.max(), "Mínimo:", rdd.min())

Total de elementos: 20
Primeros 5: [1, 2, 3, 4, 5]
Suma: 210
Máximo: 20 Mínimo: 1


In [11]:
# E3: Map Filter

numeros = sc.parallelize(range(1, 51))
por3 = numeros.map(lambda x: x * 3)            
mayores50 = por3.filter(lambda x: x > 50)      # transformación
print("Total > 50:", mayores50.count())
print("Primeros 10:", mayores50.take(10))

Total > 50: 34
Primeros 10: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]


In [12]:
# E4: Clave-Valor
ventas = [("Laptop", 1200), ("Mouse", 25), ("Laptop", 1100),
          ("Teclado", 75), ("Mouse", 30), ("Laptop", 1300)]
rdd_ventas = sc.parallelize(ventas)
total = rdd_ventas.reduceByKey(lambda a, b: a + b)
print("Total por producto:", total.collect())
conteo = rdd_ventas.mapValues(lambda v: 1).reduceByKey(lambda a, b: a + b)
print("Número de ventas:", conteo.collect())
print("Mayor total:", total.sortBy(lambda x: x[1], ascending=False).first())  # ('Laptop', 3600)

Total por producto: [('Teclado', 75), ('Mouse', 55), ('Laptop', 3600)]
Número de ventas: [('Teclado', 1), ('Mouse', 2), ('Laptop', 3)]
Mayor total: ('Laptop', 3600)


In [13]:
# Ejercicio 5: CSV
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv" -P ./DATOS
import os
path_final = "file://" + os.path.abspath("./DATOS/persons.csv") # lo que veiamos, de que fuera local.
rdd = sc.textFile(path_final)
header = rdd.first()
sin_header = rdd.filter(lambda l: l != header)
nombre_edad = sin_header.map(lambda l: l.split(",")).map(lambda c: (c[0], int(c[1])))
mayores30 = nombre_edad.filter(lambda x: x[1] > 30)
print(mayores30.collect())
print("Total:", mayores30.count())

--2026-11-19 00:31:13--  https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/persons.csv [following]
--2026-11-19 00:31:14--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/persons.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607 [text/plain]
Saving to: ‘./DATOS/persons.csv’

persons.csv         100%[===================>]     607  --.-KB/s    in 0s      

2026-11-19 00:31:14 (18.8 MB/s) - ‘./DATOS/persons.csv’ saved [607/607]

[('David Johnson', 35), ('Michael Wilson', 4

In [14]:
# E6: Evitar multiples splits:
def parsear(linea):
    c = linea.split(",")          # UN solo split por línea
    return (c[0], c[1], c[2])     # (nombre, edad, ciudad/3ª columna)

datos = rdd.map(parsear)
nombres  = datos.map(lambda x: x[0])
edades   = datos.map(lambda x: x[1])
ciudades = datos.map(lambda x: x[2])


In [15]:
# E7: Debugging:
import os
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Debug").getOrCreate()
sc = spark.sparkContext                                     # Error 1: faltaba sc
path = "file://" + os.path.abspath("./DATOS/persons.csv")   # Error 2: file:// + ruta absoluta
rdd = sc.textFile(path)
for linea in rdd.take(10):                                  # Error 3: take() en vez de collect()
    print(linea)
spark.stop()                                                # Error 4: cerrar sesión

name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000
Michael Wilson,40,Male,75000
Sarah Brown,32,Female,58000
Robert Lee,29,Male,51000
Lisa Garcia,27,Female,49000
James Martinez,38,Male,70000
